In [2]:
#+++ версия 2025.04.21 
#+++ версия 2025.04.14 tg: Цифриум, команда 2 Машинисты, @codeup1054, @Kimutogo,@Eques8, @serin_1995, @saturnian1, @tyuop077
from importlib import reload
import sys, os 
import ipynbname


for i in ['utils_log','datetime','glob']:
    if i in sys.modules: del  sys.modules[i] 
    exec (f'from {i} import *')  
    
tm()

i = 0
for k in dir(ps):
    if ('_' not in k) and isinstance(getattr(ps, k), str) :
        print (f"{getattr(ps, k)}{k:^9}{ps._} ", end="")
        if (i := i+1)%8 == 0 : print("")



NB_PATH_NAME = ipynbname.path().name
LOG_SHEET_NAME = '03.local_procc'

tg(f"🚩 Start | {LOG_SHEET_NAME} ", sheet_name=LOG_SHEET_NAME, tags=NB_PATH_NAME)

tm('\n >>>')  

 *** Start at: 19:20:58 2025-04-21  ****************************************
  BBLUE     BGRAY    BGREEN    BLBLUE    BLCYAN    BLGRAY    BLGREEN   BLILAC   
 BLLBLUE  BLMAGENTA   BLRED     BLUE       BLY    BLYELLOW  BMAGENTA    BOLD    
 BORANGE    BRED       BY      BYELLOW    CYAN    DARKCYAN      E        END    
   ERR      GRAY      GREEN     LBLUE     LGRAY    LGREEN   LMAGENTA    LRED    
 MAGENTA   ORANGE    PURPLE      RED        T       TOTAL   UNDERLINE     Y     
 YELLOW      err      0:00:06.638  ₀⡄₀₀⡄₀₆.₆₃₈ 
 >>>


datetime.datetime(2025, 4, 21, 19, 21, 5, 416737)

### 01. get_prompt

In [5]:
tm()

# Упрощенная функция подготовки промпта. комбинируем промпт и транскрипт 

def get_prompt(transcript = "Что-нибудть про LLM", prompt_path = r'Подготовь конспект'):
    prompt = f'''{prompt_path} {transcript}'''
    return prompt



# тестирование 
trans_fname = 'lectures_2025/texts/Общественное движение в России 2четверь XIX в_01_large-v3.txt'  # с таймкодом
prompt_fname = r'prompts/prompt_00.txt'  # 00- промпт написан человеком, 01-03 варианты созданы человеком

with open(trans_fname, "r", encoding="utf-8")  as f:    transcript = f.read()
with open(prompt_fname, "r", encoding="utf-8") as f:   prompt = f.read()

  
prompt = get_prompt(transcript,  prompt_path =prompt)

print(prompt)

 *** Start at: 19:24:21 2025-04-21  ****************************************
Ты — помошник лектора. Преобразуй приложенный текст в учебный конспект в формате markdown. 

Структура конспекта:
1. Заголовок презентации из первой строки текста начни с '# '

Разделы начинай с '# ': 
2. План занятия с тайм кодами. 
3. Основные концепции в формате Название - определение. 
4. Ключевые персоны c указанием годов жизни в скобках и описанием исторического вклада.  
5. Сравнение концепций c описанием идей и ключевыми персонами в виде таблицы markdown. 
6. Даты ключевых событий. 
7. Чек лист для проверки из 5 вопросов и на каждый 3 ответа
    
Ответ начни с заголовка и структурированного текста:
Заверши фразой "Конец конспекта". 

**Текст транскрипта:**

[СЮДА ВСТАВИТЬ ТЕКСТ ТРАНСКРИПТА]

Конец конспекта. 

[00:00]
ЕГЭ – это просто! Общественное движение в России во второй четверти XIX века. В общественном движении второй четверти XIX века можно выделить три направления. Консервативное, либеральное 

## 06 Local Model

### 06.01 llm_proccessing(LOCAL_MODEL = '', verbose=False, prompt = "Модель расскажи о себе")

In [7]:
tm()

# --- Генератор конспектов ---

import os
from llama_cpp import Llama

# --- Определение пути к модели ---
def resolve_model_path(path: str) -> str:
    if os.path.isdir(path):
        files = [f for f in os.listdir(path) if f.endswith('.gguf')]
        if not files:
            raise FileNotFoundError("Нет .gguf файлов в папке")
        files.sort(key=lambda f: os.path.getsize(os.path.join(path, f)), reverse=True)
        return os.path.join(path, files[0])
    elif path.endswith(".gguf") and os.path.isfile(path):
        return path
    else:
        raise ValueError("Укажи путь к .gguf-файлу или папке, содержащей такой файл")

# --- Загрузка модели ---
def load_model(path: str, **kwargs) -> Llama:
    model_path = resolve_model_path(path)
    n_ctx = max(kwargs.get("n_ctx", 4096), 1024)

    print(f"🚀 Загружаем модель: {os.path.basename(model_path)}, контекст: {n_ctx}")

    return Llama(
        model_path=model_path,
        n_ctx=n_ctx,
        n_gpu_layers=kwargs.get("n_gpu_layers", 0),
        n_batch=kwargs.get("n_batch", 64),
        n_threads=kwargs.get("n_threads", 8),
        use_mlock=kwargs.get("use_mlock", False),
        use_mmap=kwargs.get("use_mmap", True),
        verbose=False
    )

# --- Генерация конспектов ---
def generate_conspect(llm: Llama, prompt: str, save_path=None, verbose=False, **kwargs) -> str:
    n_ctx = llm.context_params.n_ctx
    reserve_tokens = min(kwargs.get("reserve_tokens", n_ctx // 2), n_ctx - 1)

    prompt_tokens = llm.tokenize(prompt.encode('utf-8'))
    available_tokens = n_ctx - len(prompt_tokens) - 10
    max_tokens = min(kwargs.get("max_tokens", n_ctx - 512), available_tokens)

    if verbose:
        print(f"\n📥 Промпт: [{len(prompt)}] {ps.BLYELLOW} {prompt} {ps._}")
        print(f"🔢 Токенов в промпте: {len(prompt_tokens)}")
        print(f"🧠 Будет сгенерировано (max_tokens): {max_tokens}")
        
        tg(f'📜 В промпте:{len(prompt)} | 🔢 токенов в промпте: {len(prompt_tokens)} | 🧠 в ответе (max_tokens): {max_tokens}')

    stream = llm.create_completion(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=kwargs.get("temperature", 0.7),
        top_p=kwargs.get("top_p", 0.9),
        stop=kwargs.get("stop"),
        stream=True
    )

    result = ""

    for n, chunk in enumerate(stream):
        token = chunk.get("choices", [{}])[0].get("text", "")
        print(token, end="", flush=True)
        result += token

        if save_path:
            with open(save_path, "w", encoding="utf-8") as f:
                f.write(result)
                f.flush()
        
        if n % 100 == 1 and n > 1:
           tg(f'({len(result):>6}) {result[-100:].replace ('\n',' ')}', tags=f'{len(token)}')
           tm(f'✍️ Результат len = {len(result)} ')

    return result.strip()

tm('>>>')

 *** Start at: 19:25:50 2025-04-21  ****************************************
  0:00:00.887  ₀⡄₀₀⡄₀₀.₈₈₇ >>>


datetime.datetime(2025, 4, 21, 19, 25, 51, 102280)

In [9]:
# Перечень моделей скачанных с использованием LM Studio

_models = [
r'C:\\!models\\reedmayhew\\Llama-3.1-8B-claude-3.7-sonnet-reasoning-distilled\\llama-3.1-8b-claude-3.7-sonnet-reasoning-distilled.Q4_0.gguf',
r'C:\\!models\\DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf',
r'C:\\!models\\Mistral-7B-Instruct-v0.3.Q8_0.gguf',
r'C:\\!models\\DeepSeek-R1-Distill-Qwen-14B-IQ4_XS.gguf',
r'C:\\!models\\reedmayhew\\Llama-3.1-8B-claude-3.7-sonnet-reasoning-distilled\\llama-3.1-8b-claude-3.7-sonnet-reasoning-distilled.Q8_0.gguf',
r'C:\\!models\\claude-3.7-sonnet-reasoning-gemma3-12B.Q8_0.gguf',
r'C:\\!models\\Mistral-7B-Instruct-v0.3.fp16.gguf',
r'C:\\!models\\gemma-3-27B-it-qat-GGUF\\gemma-3-27B-it-QAT-Q4_0.gguf'
]

_models = [
'C:\\!models\\mradermacher\\Llama4Some-SOVL-4x8B-L3-V1-i1-GGUF\\Llama4Some-SOVL-4x8B-L3-V1.i1-Q4_K_S.gguf',
'C:\\!models\\reedmayhew\\Llama-3.1-8B-claude-3.7-sonnet-reasoning-distilled\\llama-3.1-8b-claude-3.7-sonnet-reasoning-distilled.Q8_0.gguf',
'C:\\!models\\bartowski\\Phi-3-medium-128k-instruct-GGUF\\Phi-3-medium-128k-instruct-Q4_K_S.gguf',
'C:\\!models\\Kondara\\DeepSeek-R1-Distill-Qwen-7B-Q4_K_M-GGUF\\deepseek-r1-distill-qwen-7b-q4_k_m.gguf',
'C:\\!models\\reedmayhew\\Llama-3.1-8B-claude-3.7-sonnet-reasoning-distilled\\llama-3.1-8b-claude-3.7-sonnet-reasoning-distilled.Q4_0.gguf',
'C:\\!models\\lmstudio-community\\gemma-3-1B-it-qat-GGUF\\gemma-3-1B-it-QAT-Q4_0.gguf',
'C:\\!models\\pszemraj\\flan-t5-large-grammar-synthesis\\ggml-model-Q6_K.gguf',
          ]

tm(f'Models len={len(_models)}')

  0:00:10.304  ₀⡄₀₀⡄₁₁.₁₉₂ Models len=7


datetime.datetime(2025, 4, 21, 19, 26, 1, 407131)

In [13]:
# токеназайер оптимизирует промпт для модели 

def analyze_prompt_tokens(llm, prompt, n_ctx=4096, reserve_tokens=None, verbose=True):
    """
    Анализ токенов перед генерацией: сколько токенов в промпте, сколько можно сгенерировать.
    """
    prompt_tokens = llm.tokenize(prompt.encode('utf-8'))
    token_count = len(prompt_tokens)
    reserve_tokens = reserve_tokens or n_ctx // 2
    reserve_tokens = min(reserve_tokens, n_ctx - 1)

    available_tokens = n_ctx - token_count - 10
    max_tokens = min(reserve_tokens, available_tokens)

    if verbose:
        print(f"🔍 Анализ промпта: {len(prompt)}")
        print(f"  📏 Всего токенов в контексте: {n_ctx}")
        print(f"  🧾 Токенов в промпте: {token_count}")
        print(f"  🧠 Можно сгенерировать: {max_tokens} токенов\n")

        if token_count + max_tokens > n_ctx:
            print("⚠️ Возможен выход за пределы контекста! Уменьши prompt или max_tokens")

    return {
        "prompt_tokens": token_count,
        "available_tokens": available_tokens,
        "max_tokens": max_tokens
    }


llm = load_model(_models[0], n_ctx=8192, n_threads=6)


with open(trans_fname, "r", encoding="utf-8")  as f:   transcript = f.read()
with open(prompt_fname, "r", encoding="utf-8") as f:   prompt = f.read()


prompt = get_prompt(prompt,transcript)

stats = analyze_prompt_tokens(llm, prompt, n_ctx=8000, verbose=True)
tm('>>>')
stats

🚀 Загружаем модель: Llama4Some-SOVL-4x8B-L3-V1.i1-Q4_K_S.gguf, контекст: 8192
🔍 Анализ промпта: 6517
  📏 Всего токенов в контексте: 8000
  🧾 Токенов в промпте: 2027
  🧠 Можно сгенерировать: 4000 токенов

  0:00:14.189  ₀⡄₀₁⡄₀₁.₈₃₉ >>>


{'prompt_tokens': 2027, 'available_tokens': 5963, 'max_tokens': 4000}

### 06.01 Обработка для 4-ех вариантов промптов всех модедей из _models 
Logging результатов обработки в google_sheet для дальнейшего анализа 

In [27]:
import glob
# --- Получить размер файла/папки ---
def get_size(path):
    if os.path.isfile(path):
        return os.path.getsize(path)
    elif os.path.isdir(path):
        total = 0
        for dirpath, _, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if os.path.isfile(fp):
                    total += os.path.getsize(fp)
        return total
    else:
        raise FileNotFoundError(f"Путь не найден: {path}")


tm()


if __name__ == "__main__":
    tg(f"🚩🚩🚩 Start ({len(_models)}) model proccessing", lap_id='lap1', reset=True)


    for p, prompt_fname in enumerate(glob.glob('prompts/pro*.txt')):


        with open(prompt_fname, "r", encoding="utf-8") as f:
            prompt_tpl = f.read()
            prompt_tpl = prompt_tpl.replace('[СЮДА ВСТАВИТЬ ТЕКСТ ТРАНСКРИПТА]', '')

        transcript_fname = r'lectures_2025\texts\Общественное движение в России 2четверь XIX в_01_large.txt'
        with open(transcript_fname, "r", encoding="utf-8") as f:
            transcript = f.read()
        
        _prompt = get_prompt(transcript, prompt_tpl)
        
        tg(f"🚩📜Next prompt ({p:^3}/{len(_prompt)})", lap_id='lap1', tags=f'📜({p})'
        
    
        for n, path_to_model in enumerate(_models[:]):
            model_size = get_size(path_to_model) / (1024 * 1024)
            
            print(f"🚩🚩🚩 ({n:<3}): ({model_size:.2f}) MB {path_to_model}")
            
            tg(f"🚩🌌Next model({n:^3}/{len(_models)})", lap_id='lap1', reset=True)
            
            
            # path_to_model = r"C:\!models\gemma-3-27B-it-qat-GGUF"
            output_dir = "C:/!conspects/"
    
            try:
                llm = load_model(path_to_model, n_ctx=4096, n_threads=6)
    
        
                stats = analyze_prompt_tokens(llm, prompt, n_ctx=9500, verbose=True)
                
            
                tg(f"🌌 ({model_size:.2f}) MB Loaded model  {str(path_to_model)}")
                tg(f"🧠 analyze_prompt_tokens {str(stats)} ")
                
                model_name = os.path.basename(path_to_model).replace(".gguf", "").replace("/", "_").replace("\\", "_") # Убрал .gguf
                    
                file_id = f"{model_name}"
                
                save_path = os.path.join(output_dir, f"conspect_{file_id}_{p:<2.0f}.md")
                
            
                # llm = load_model(path_to_model, n_ctx=4096, n_threads=6)
            
                result = generate_conspect(
                    llm,
                    prompt,
                    max_tokens=4000,
                    temperature=0.6,
                    save_path=save_path,
                    stop=["Конец"],
                    verbose=False,
                )
        
            except Exception as e:
                   tg(f"📛📛📛 err = {e} ({model_size:.2f}) MB Loaded model  {str(path_to_model)}")         
            
            tg(f"✅ ({n:<3}) | Результат: {len(result)} | {path_to_model}", tags = f"{len(result)}")

tg(f'✅ Обработано {len(_models)}')
tm('>>>')


 *** Start at: 20:47:21 2025-04-20  ****************************************
🚩🚩🚩 (0  ): (13632.28) MB C:\!models\mradermacher\Llama4Some-SOVL-4x8B-L3-V1-i1-GGUF\Llama4Some-SOVL-4x8B-L3-V1.i1-Q4_K_S.gguf
🚀 Загружаем модель: Llama4Some-SOVL-4x8B-L3-V1.i1-Q4_K_S.gguf, контекст: 4096


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


🔍 Анализ промпта:
  📏 Всего токенов в контексте: 9500
  🧾 Токенов в промпте: 1979
  🧠 Можно сгенерировать: 4750 токенов

7 марта 1849 года. 

[07:35]
В России в XIX веке существовало несколько направлений общественного движения, которые отличались своей идеологией и методами. Консервативное, либеральное и радикальное направления. В каждом из них были свои идеологи, которые в свою очередь влияли на развитие общества. 

**🚩🚩🚩 (1  ): (8145.12) MB C:\!models\reedmayhew\Llama-3.1-8B-claude-3.7-sonnet-reasoning-distilled\llama-3.1-8b-claude-3.7-sonnet-reasoning-distilled.Q8_0.gguf
🚀 Загружаем модель: llama-3.1-8b-claude-3.7-sonnet-reasoning-distilled.Q8_0.gguf, контекст: 4096


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


🔍 Анализ промпта:
  📏 Всего токенов в контексте: 9500
  🧾 Токенов в промпте: 1979
  🧠 Можно сгенерировать: 4750 токенов

.


# Общественное движение в России второй половины XIX века

## План занятия

1. Введение в тему
2. Консервативное направление (Уваров и слуга)
3. Либеральное направление (славянофилы и западники)
4. Революционное движение
5. Заключение и вопросы

## Основные концепции

- **Консерват  0:04:08.346  ₀⡄₀₄⡄₀₈.₃₄₆ ✍️ Результат len = 286 
изм** - сохранение traditionalных институтов и ценностей
- **Либерализм** - расширение индивидуальных прав и ограничение государственной власти
- **Революционизм** - радикальное изменение социальной системы

## Ключевые персоны

| Имя | Годы жизни | Исторический вклад |
| --- | --- | --- |
| Сергей Уваров | 1786  0:00:27.510  ₀⡄₀₄⡄₃₅.₈₅₆ ✍️ Результат len = 599 
-1855 | Разработал теорию официальной народности и был идеологом консерватизма |
| Александр Хомяков | 1804-1860 | Основоположник славянофильства и сторонник национального развит

...

llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


🔍 Анализ промпта:
  📏 Всего токенов в контексте: 9500
  🧾 Токенов в промпте: 1807
  🧠 Можно сгенерировать: 4750 токенов



[07:32]
В 1850-е годы, в период правления императора Александра II, в России начались реформы, направленные на ограничение власти монарха и развитие экономики. Эти реформы, в частности, реформа землевладения, реформа образования и реформа промышленности, стали важной вехой в истории России.

[08:00]
🚩🚩🚩 (6  ): (614.05) MB C:\!models\pszemraj\flan-t5-large-grammar-synthesis\ggml-model-Q6_K.gguf
🚀 Загружаем модель: ggml-model-Q6_K.gguf, контекст: 4096


llama_init_from_model: n_ctx_pre_seq (4096) > n_ctx_train (512) -- possible training context overflow


🔍 Анализ промпта:
  📏 Всего токенов в контексте: 9500
  🧾 Токенов в промпте: 4513
  🧠 Можно сгенерировать: 4750 токенов

  0:00:13.630  ₂⡄₃₂⡄₀₃.₄₃₆ >>>


datetime.datetime(2025, 4, 20, 23, 19, 24, 568729)

In [5]:
import os
from llama_cpp import Llama

def resolve_model_path(path: str) -> str:
    """Если path — папка, выбираем самый большой .gguf файл в ней"""
    if os.path.isdir(path):
        files = [f for f in os.listdir(path) if f.endswith('.gguf')]
        if not files:
            raise FileNotFoundError("Нет .gguf файлов в папке")
        files.sort(key=lambda f: os.path.getsize(os.path.join(path, f)), reverse=True)
        return os.path.join(path, files[0])
    elif path.endswith(".gguf") and os.path.isfile(path):
        return path
    else:
        raise ValueError("Укажи путь к .gguf-файлу или папке, содержащей такой файл")

def llm_processing(LOCAL_MODEL='', prompt="Модель, расскажи о себе", verbose=False, **kwargs):
    # --- Путь к модели ---
    model_path = resolve_model_path(LOCAL_MODEL)

    # --- Параметры по умолчанию ---
    n_ctx = kwargs.get("n_ctx", 4096)
    n_ctx = max(n_ctx, 1024)  # минимум
    params = {
        "model_path": model_path,
        "n_ctx": n_ctx,
        "n_gpu_layers": kwargs.get("n_gpu_layers", 0),
        "n_batch": kwargs.get("n_batch", 64),
        "n_threads": kwargs.get("n_threads", 8),
        "use_mlock": kwargs.get("use_mlock", False),
        "use_mmap": kwargs.get("use_mmap", True),
        "verbose": False
    }

    # --- Инициализация LLaMA ---
    print(f"🚀 Загружаем модель: {os.path.basename(model_path)}, контекст: {n_ctx}")
    llm = Llama(**params)

    # --- Подготовка промпта ---
    reserve_tokens = min(kwargs.get("reserve_tokens", n_ctx // 2), n_ctx - 1)
    prompt_tokens = llm.tokenize(prompt.encode('utf-8'))
    available_tokens = n_ctx - len(prompt_tokens) - 10
    max_tokens = min(kwargs.get("max_tokens", n_ctx - 512), available_tokens)

    if verbose:
        print(f"\n📥 Промпт: {ps.BLBLUE} {prompt}{ps._}\n")
        print(f"🔢 Токенов в промпте: {len(prompt_tokens)}")
        print(f"🧠 Будет сгенерировано (max_tokens): {max_tokens}")

    # --- Генерация ---
    stream = llm.create_completion(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=kwargs.get("temperature", 0.7),
        top_p=kwargs.get("top_p", 0.9),
        stop=kwargs.get("stop"),
        stream=True
    )

    result = ""

    def format_value(v):
        if isinstance(v, float): return f"{v:.2f}".replace(".", "_")
        elif isinstance(v, bool): return "T" if v else "F" # Короче
        # Убираем недопустимые символы для имен файлов
        s = str(v).replace(" ", "_").replace(":", "-").replace("\\", "_").replace("/", "_")
        s = ''.join(c for c in s if c.isalnum() or c in ('_', '-'))
        return s[:20] # Ограничение длины
        

    output_dir = "C:/!conspects/"

    # kwarg_str = "__".join(f"{k}-{format_value(v)}" for k, v in fname_params.items())

    kwarg_str = ""

    model_name = os.path.basename(LOCAL_MODEL).replace(".gguf", "").replace("/", "_").replace("\\", "_") # Убрал .gguf
    
    
    file_id = f"{model_name}__{kwarg_str}"

    save_path = os.path.join(output_dir, f"conspect_{file_id}.md")
    
    for n, chunk in enumerate(stream):
        token = chunk.get("choices", [{}])[0].get("text", "")
        print(token, end="", flush=True)
        result += token

        with open(save_path, "w", encoding="utf-8") as f:
            print(token, end="", flush=True) # Вывод в консоль
            f.write(token) # Запись в файл
            f.flush()
            
            chars_written_total = len(result)
            
            if n % 100 == 1 and n > 1:
               tg(f'✍️ ({chars_written_total:>6}) {save_path}')
               tm(f'✍️ ({chars_written_total:>6}) {save_path}')
    

    return result.strip()


def get_size(path):
    if os.path.isfile(path):
        return os.path.getsize(path)  # размер одного файла
    elif os.path.isdir(path):
        total_size = 0
        for dirpath, dirnames, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if os.path.isfile(fp):
                    total_size += os.path.getsize(fp)
        return total_size
    else:
        raise FileNotFoundError(f"Путь не найден: {path}")


if __name__ == "__main__":
    path_to_model = r"C:\!models\gemma-3-27B-it-qat-GGUF"  # Можно указать папку
    
    
    prompt = get_prompt(transcript)

    model_size = get_size(path_to_model) / (1024 * 1024)
    
    tm(f' ({model_size:>6.2f}) MB 🌌 {path_to_model}')
    tg(f' ({model_size:>6.2f}) MB 🌌 {path_to_model}')
    
    result = llm_processing(
        LOCAL_MODEL=path_to_model,
        prompt=prompt,
        max_tokens=5048,
        temperature=0.6,
        verbose=True
    )

    print("\n\n✅ Результат:\n", result)

tg(f'✅ {len(result)}  | {path_to_model}')

  0:03:02.977  ₀⡄₁₅⡄₁₄.₂₆₅  (15664.23) MB 🌌 C:\!models\gemma-3-27B-it-qat-GGUF
🚀 Загружаем модель: gemma-3-27B-it-QAT-Q4_0.gguf, контекст: 4096


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



📥 Промпт:  
Ты — помошник лектора. Преобразуй приложенный текст в учебный конспект в формате markdown. 

Структура конспекта:
1. Заголовок презентации из первой строки текста начни с '# '

Разделы начинай с '# ': 
2. План занятия с тайм кодами. 
3. Основные концепции в формате Название - определение. 
4. Ключевые персоны c указанием годов жизни в скобках и описанием исторического вклада.  
5. Сравнение концепций c описанием идей и ключевыми персонами в виде таблицы markdown. 
6. Даты ключевых событий. 
7. Чек лист для проверки из 5 вопросов и на каждый 3 ответа
    
Ответ начни с заголовка и структурированного текста:
Заверши фразой "Конец конспекта". 

Используй приложенный текст:
[00:00]
ЕГЭ – это просто! Общественное движение в России во второй четверти XIX века. В общественном движении второй четверти XIX века можно выделить три направления. Консервативное, либеральное и радикальное. Идеологом консерватизма стал Сергей Семенович Уваров. Теория официальной народности, разработанная

NameError: name 'n' is not defined

In [46]:
import os
# import time # Убрано, предполагается, что используется в tm из utils_log
from llama_cpp import Llama, llama_n_ctx
from contextlib import redirect_stdout, redirect_stderr
import traceback # Оставляем для детального вывода ошибок

# ПРЕДПОЛАГАЕТСЯ, ЧТО ВЫ ИМПОРТИРОВАЛИ tm, tg И МОДУЛЬ/ОБЪЕКТ ps ИЗ utils_log.py ВЫШЕ
# Пример:
# from utils_log import tm, tg, ps # Или как он у вас называется

def truncate_prompt(llm, prompt, reserve_tokens=2048):
    # Важно: tokenize может потребовать корректную загрузку модели,
    # поэтому его лучше вызывать *после* инициализации llm
    try:
        prompt_tokens = llm.tokenize(prompt.encode("utf-8"))
        # Убедимся, что у llm есть контекст перед вызовом llama_n_ctx
        if llm.ctx is None:
            print("Предупреждение: Контекст модели еще не инициализирован для truncate_prompt.")
            # Возвращаем как есть или применяем базовую обрезку
            return prompt[:8000] # Пример ограничения по символам

        max_tokens = llama_n_ctx(llm.ctx) - reserve_tokens

        if len(prompt_tokens) > max_tokens:
            prompt_tokens = prompt_tokens[-max_tokens:]
            prompt = llm.detokenize(prompt_tokens).decode("utf-8", errors="ignore")
            print(f"Промпт был усечен до {len(prompt_tokens)} токенов.") # Информационное сообщение
    except Exception as e:
        print(f"Предупреждение: Не удалось обрезать промпт из-за ошибки токенизации: {e}")
        # Возвращаем исходный промпт или применяем простую обрезку по символам
        # Можно получить n_ctx из параметров, если llm.ctx еще не готов
        # max_chars_approx = (params.get('n_ctx', 2048) - reserve_tokens) * 3 # Очень грубая оценка
        # if len(prompt) > max_chars_approx:
        #     prompt = prompt[-max_chars_approx:]
        # Пока просто вернем как есть при ошибке
        pass # Вернем оригинальный промпт при ошибке токенизации
    return prompt


# --- Обновленная функция без заглушек tm, tg, ps ---
def llm_proccessing(LOCAL_MODEL='', prompt="Модель, расскажи о себе", verbose=False, **kwargs):
    # print(f'Prompt 1 📜 ({len(prompt)} chars):\n {ps.BLBLUE} {prompt[:]}... {ps._} \n')

    # Проверяем, что основной путь указан и файл существует
    if not LOCAL_MODEL:
        raise ValueError("Необходимо указать путь к файлу модели (LOCAL_MODEL)")
    if not os.path.exists(LOCAL_MODEL):
        raise FileNotFoundError(f"Файл модели не найден по пути: {LOCAL_MODEL}")

    # --- Автоматическое определение пути к CLIP модели ---
    model_dir = os.path.dirname(LOCAL_MODEL)
    clip_filename = "mmproj-model-f16.gguf"
    potential_clip_path = os.path.join(model_dir, clip_filename)
    clip_model_path_to_load = None

    if os.path.exists(potential_clip_path):
        clip_model_path_to_load = potential_clip_path
        tm(f"🔍 Найден файл CLIP модели: {clip_filename}") # Используем tm из utils_log
    else:
        tm(f"ℹ️ Файл CLIP модели '{clip_filename}' не найден в директории {model_dir}. Загрузка как текстовой модели.") # Используем tm из utils_log
    # --- Конец автоопределения ---

    # Параметры модели
    default_n_ctx = 4096
    params = {
        "n_ctx": kwargs.get("n_ctx", default_n_ctx),
        "n_gpu_layers": kwargs.get("n_gpu_layers", 0),
        "n_batch": kwargs.get("n_batch", 64), # Можно попробовать уменьшить до 32 или 16 если не хватает VRAM/RAM
        "n_threads": kwargs.get("n_threads", 8),
        "use_mlock": kwargs.get("use_mlock", False),
        "use_mmap": kwargs.get("use_mmap", True),
        "verbose": False # verbose для llama_cpp
    }

    # Аргументы для конструктора Llama
    llama_args = {
        "model_path": LOCAL_MODEL,
        **params
    }
    if clip_model_path_to_load:
        llama_args["clip_model_path"] = clip_model_path_to_load

    # Инициализация модели
    llm = None
    try:
        tm(f"⏳ Загрузка модели: {os.path.basename(LOCAL_MODEL)}" + (f" + {clip_filename}" if clip_model_path_to_load else "")) # Используем tm
        # Если нужна тихая загрузка без вывода llama.cpp:
        # with open(os.devnull, "w") as fnull:
        #    with redirect_stdout(fnull), redirect_stderr(fnull):
        #        llm = Llama(**llama_args)
        # Иначе (для отладки):
        llm = Llama(**llama_args)

        # Формируем сообщение об успешной загрузке (без цветов ps.*)
        info_msg = f"Llama({os.path.basename(LOCAL_MODEL)}"
        if clip_model_path_to_load:
            info_msg += f" + {clip_filename}"
        info_msg += f")"
        tm(f'🌌 Модель загружена: {info_msg}') # Используем tm

    except Exception as e:
        print(f"Ошибка при загрузке модели:") # Убраны цвета ps.*
        traceback.print_exc()
        raise

    # Обрезка промпта (теперь после инициализации llm)
    full_prompt = truncate_prompt(llm, prompt, reserve_tokens=kwargs.get("reserve_tokens", 2048))

    # print (f'🎈verbose={verbose}')

    if verbose:
        # Вывод промпта без цветов ps.*
        print(f'Full_Prompt 🎈 📜 ({len(full_prompt)} chars):\n {ps.BLYELLOW} {full_prompt[:]}... {ps._} \n')

    # Генерация
    max_tokens = kwargs.get("max_tokens", 6048)
    temperature = kwargs.get("temperature", 0.5)
    top_p = kwargs.get("top_p", 0.95)
    # Можно добавить стоп-токены по умолчанию для llama3/gemma3, если нужно
    default_stop = ["<|eot_id|>", "<|end_of_turn|>", "<|endoftext|>"]
    stop = kwargs.get("stop", default_stop)

    result_text = ""

    # Подготовка имени файла
    model_name = os.path.basename(LOCAL_MODEL).replace(".gguf", "").replace("/", "_").replace("\\", "_") # Убрал .gguf
    def format_value(v):
        if isinstance(v, float): return f"{v:.2f}".replace(".", "_")
        elif isinstance(v, bool): return "T" if v else "F" # Короче
        # Убираем недопустимые символы для имен файлов
        s = str(v).replace(" ", "_").replace(":", "-").replace("\\", "_").replace("/", "_")
        s = ''.join(c for c in s if c.isalnum() or c in ('_', '-'))
        return s[:20] # Ограничение длины

    fname_params = {
        'ctx': params['n_ctx'],
        'temp': temperature,
        'top_p': top_p,
        'max_t': max_tokens
    }
    kwarg_str = "__".join(f"{k}-{format_value(v)}" for k, v in fname_params.items())
    
    # Сделаем имя короче: уберем model_name если он слишком длинный
    
    base_name_part = model_name[:30] # Ограничим длину имени модели в файле
    file_id = f"{base_name_part}__{kwarg_str}"


    # Пути сохранения
    SSD_PATH = "C:/!conspects/"
    HDD_PATH = "conspects/"
    output_dir = SSD_PATH if os.path.exists(SSD_PATH) else HDD_PATH
    if not os.path.exists(output_dir):
        try:
            os.makedirs(output_dir)
            tm(f"Создана директория для конспектов: {output_dir}")
        except OSError as e:
            print(f"Не удалось создать директорию {output_dir}: {e}")
            # В случае ошибки используем текущую директорию
            output_dir = "."

    save_path = os.path.join(output_dir, f"conspect_{file_id}.md")

    # Стриминг и сохранение
    tm(f"🚀 Генерация ответа (сохранение в {save_path}):") # Используем tm, убраны цвета ps.*
    stream = llm.create_completion(
        prompt=full_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        stop=stop,
        stream=True
    )

    chars_written_total = 0
    
    try:
        # сохраняем промпт
        with open(save_path.replace(".md", "_prompt.md"), "w", encoding="utf-8") as f:
            # Записываем метаданные и промпт в начало файла
            f.write(f"# Model: {os.path.basename(LOCAL_MODEL)}\n")
            if clip_model_path_to_load:
                f.write(f"# CLIP Model: {clip_filename}\n")
            f.write(f"# Params: {fname_params}\n")
            f.write(f"# Stop Tokens: {stop}\n\n")
            f.write(f"## Prompt:\n\n```\n{prompt}\n```\n\n { "="*80 } Response { "="*80 } \n\n")


        with open(save_path, "w", encoding="utf-8") as f:
            for n, response in enumerate(stream):
                if "choices" not in response or not response["choices"]:
                    continue
                token = response["choices"][0].get("text", "")
                if token is None: token = ""
    
                result_text += token
                print(token, end="", flush=True) # Вывод в консоль
                f.write(token) # Запись в файл
                
                f.flush()
                
                chars_written_total += len(token)
    
                # Логгирование каждые N символов, а не токенов (более стабильно)
                # Используем tg из utils_log
                
                if n % 100 == 1 and n > 1:
                   tg(f'✍️ ({chars_written_total:>6}) {save_path}')
                   tm(f'✍️ ({chars_written_total:>6}) {save_path}')

    except Exception as e:
        print(f"\nОшибка во время стриминга или записи файла:") # Убраны цвета ps.*
        traceback.print_exc()
    finally:
         print() # Перевод строки после окончания генерации
         # Используем tm из utils_log, убраны цвета ps.*
         tm(f"\n✅ Конспект сохранён: {chars_written_total} символов в {save_path} >>>")

    return result_text.strip()

# --- Пример вызова ---
if __name__ == "__main__":
    # ПРЕДПОЛАГАЕТСЯ, ЧТО ЗДЕСЬ ИЛИ ГЛОБАЛЬНО ИМПОРТИРОВАНЫ tm, tg, ps из utils_log.py
    # from utils_log import tm, tg, ps # Пример

    main_model_path = r'C:\!models\gemma-3-27B-it-qat-GGUF\gemma-3-27B-it-QAT-Q4_0.gguf'

    if not os.path.exists(main_model_path):
        print(f"Ошибка: Файл основной модели не найден: {main_model_path}") # Убраны цвета ps.*
    else:
        try:
            result = llm_proccessing(
                LOCAL_MODEL=main_model_path,
                prompt=get_prompt(transcript),   #"Напиши короткий рассказ о космическом путешественнике, который нашел разумное растение.",
                max_tokens=500,
                temperature=0.7,
                top_p=0.9,
                verbose=True
            )
            # print("\n--- Итоговый результат ---")
            # print(result)
        except FileNotFoundError as e:
             print(f"\nОшибка: {e}") # Убраны цвета ps.*
        except ValueError as e:
             print(f"\nОшибка: {e}") # Убраны цвета ps.*
        except Exception as e:
            print(f"\nПроизошла непредвиденная ошибка во время обработки.") # Убраны цвета ps.*
            traceback.print_exc()

    tm('Готово 🚀') # Используем tm из utils_log

  0:02:49.638  ₃⡄₂₃⡄₅₇.₄₅₆ 🔍 Найден файл CLIP модели: mmproj-model-f16.gguf
         0:00  ₃⡄₂₃⡄₅₇.₄₅₆ ⏳ Загрузка модели: gemma-3-27B-it-QAT-Q4_0.gguf + mmproj-model-f16.gguf


llama_init_from_model: n_ctx_per_seq (5056) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


  0:00:59.402  ₃⡄₂₄⡄₅₆.₈₅₉ 🌌 Модель загружена: Llama(gemma-3-27B-it-QAT-Q4_0.gguf + mmproj-model-f16.gguf)
Промпт был усечен до 766 токенов.
Full_Prompt 🎈 📜 (2653 chars):
  ли, что Россия должна двигаться в общеевропейском направлении, сменить самодержавие на конституционную монархию, расширять права всех сословий, развивать рыночную экономику и ввести парламент как орган волеизъявления народа. 

[04:01]
Основными представителями западников были Тимофей Николаевич Грановский, Сергей Михайлович Соловьев, Константин Дмитриевич Кавелин, Павел Васильевич Аненков, Василий Петрович Боткин, Иван Сергеевич Тургенев. И западники и славянофилы, единственным возможным путем видели реформы, организуемые и проводимые властью. На формирование западнической и славянофильской идеологии 

[04:32]
значительное влияние оказал Петр Яковлевич Чаадаев, публицист и мыслитель. В философических письмах он говорил об «отлученности России от всемирной истории, духовном застое в России, который препятствует ее ис

### 06.01 Antropic claude-3.7-sonnet-reasoning-gemma3-12B.Q8_0

### 06.03 Mistral-7B-Instruct-v0.3.fp16.gguf

In [32]:
tm()
LOCAL_MODEL = r'models/Mistral-7B-Instruct-v0.3.fp16.gguf'

prompt = f"{get_prompt(transcript)}"

result_text = llm_proccessing(LOCAL_MODEL = LOCAL_MODEL, prompt = prompt, verbose = True)
tm(f"\n finish 🎁: {len(result_text)} символов >>>")  

 *** Start at: 21:14:22 2025-04-16  ************************************************************
  0:00:00.001  ₀⡄₀₀⡄₀₀.₀₀₁ 01. 📄 Файл: lectures_2025/Общественное движение в России 2четверь XIX в_01_large.txt — 5580 символов | Модель: models/Mistral-7B-Instruct-v0.3.fp16.gguf
Prompt 📃📃📃
 
Ты — помощник лектора, готовишь конспект. Преобразуй приложенный текст в структурированный учебный конспект в формате markdown. 

Структура конспекта:
1. Заголовок презентации:  из первой строки приложенного текста начни с #

Разделы начинай с #: 
2. План занятия с тайм кодами, 
3. Основные концепции в формате Название - определение, 
4. Ключевые персоны c указанием годов жизни в скобках и описанием исторического вклада. 5. Проверяй в своей базе фамилии и имена участников, относящихся к описываемым  
5. Сравнение концепций c описанием идей в виде таблицы markdown, 
6. Даты ключевых событий, 
7. Чек лист для проверки из 5 вопросов и на каждый 3 ответа
    
Ответ начни с заголовка и структурированного т

datetime.datetime(2025, 4, 16, 21, 40, 47, 948137)

### 06.03 Deepseek 

In [50]:
tm()
LOCAL_MODEL = r'C:\!haggingface\transformers\DeepSeek-R1-Distill-Qwen-14B-GGUF\DeepSeek-R1-Distill-Qwen-14B-IQ4_XS.gguf'

prompt = f"{get_prompt(transcript)}"

result_text = llm_proccessing(LOCAL_MODEL = LOCAL_MODEL, prompt = prompt, verbose = True)
tm(f"\n finish 🎁: {len(result_text)} символов >>>")  

 *** Start at: 17:57:56 2025-04-16  ************************************************************
         0:00         ₀⡄₀₀ 01. 📄 Файл: lectures_2025/Общественное движение в России 2четверь XIX в_01_large.txt — 5580 символов | Модель: C:\!haggingface\transformers\DeepSeek-R1-Distill-Qwen-14B-GGUF\DeepSeek-R1-Distill-Qwen-14B-IQ4_XS.gguf
  0:00:09.577  ₀⡄₀₀⡄₀₉.₅₇₇ 02. 🚀 Модель загружена: Llama(C:\!haggingface\transformers\DeepSeek-R1-Distill-Qwen-14B-GGUF\DeepSeek-R1-Distill-Qwen-14B-IQ4_XS.gguf)
 Всего доброго!

Хорошо, я получил задание преобразовать текст лекции в структурированный учебный конспект в формате markdown. Сначала нужно понять, что от меня требуют. Пользователь дал подробные указания: разделы с таймкодами, основные концепции в виде названий и определений, ключевые персоны с датами жизни и их вкладом, таблицы для сравнения концепций, даты ключевых событий и чек-лист для проверки.

Начну с анализа исходного текста. В нём говорится о ЕГЭ и общественном движении во второй чет

datetime.datetime(2025, 4, 16, 18, 13, 33, 939445)

### 06.03 meta-llama-3-8b-instruct.Q4_K_M

In [20]:
tm()
LOCAL_MODEL = r'models/Meta-Llama-3.1-8B-Instruct.Q8_0.gguf' #meta-llama-3-8b-instruct.Q4_K_M.gguf'

result_text = llm_proccessing(LOCAL_MODEL = LOCAL_MODEL, verbose=True)
tm(f"\finish 🎁 >>>")

 *** Start at: 19:06:37 2025-04-16  ************************************************************
         0:00         ₀⡄₀₀ 01. 📄 Файл: lectures_2025/Общественное движение в России 2четверь XIX в_01_large.txt — 5580 символов | Модель: models/Meta-Llama-3.1-8B-Instruct.Q8_0.gguf
Prompt 📃📃📃
 
    Ты — помощник лектора, готовишь конспект. Преобразуй данный текст в структурированный учебный конспект в формате markdown
    Структура конспекта:
    - [00:00]
ЕГЭ – это просто! Общественное движение в России во второй четверти XIX века
    - Разделы: План занятия с тайм кодами, Основные концепции в формате Название - определение, Ключевые персоны c указанием годов жизни в скобках и описанием исторического вклада. Проверяй в своей базе фамилии и имена участников, относящихся в описываемым событиям, некоторые фамилии при транскрибации они могут быть неточно распознаны, Сравнение концепций c описанием идей в виде таблицы markdown, Даты ключевых событий, Чек лист для проверки из 5 вопросов с 3-мя 

datetime.datetime(2025, 4, 16, 19, 22, 6, 247413)

### 06.04 mistral-7b-instruct-v0.1.Q8_0.gguf

In [62]:
instruction = get_prompt(transcript)[1]['content']
instruction

'[00:00]\nЕГЭ – это просто! Общественное движение в России во второй четверти XIX века. В общественном движении второй четверти XIX века можно выделить три направления. Консервативное, либеральное и радикальное. Идеологом консерватизма стал Сергей Семенович Уваров. Теория официальной народности, разработанная Уваровым, базировалась на трех ключевых принципах – самодержавие, православие и народность. \n\n[00:35]\nКаждый из принципов имел идеологическое обоснование. Самодержавие – это основа жизни русского общества. Православие – это ориентация человека на общественный интерес, общее благо и справедливость. Народность выражала единство народа, сплоченного вокруг царя. Между народом и монархом существовала неразрывная духовная связь, которая была и будет основой успешного развития России. \n\n[01:10]\nСторонниками Уварова и идей консерватизма были Николай Герасимович Устрялов и Михаил Петрович Погодин. Писатели Фадди Венедиктович Булгарин. Николай Иванович Греч. Михаил Николаевич Загоскин

In [79]:
tm()
LOCAL_MODEL = r'models/mistral-7b-instruct-v0.1.Q8_0.gguf'

prompt = f"[INST] {get_prompt(transcript)} [/INST]"

tm (f"len prompt [[{len(prompt)}]]")

result_text = llm_proccessing(LOCAL_MODEL = LOCAL_MODEL, verbose = True, prompt = prompt)
tm(f"\finish 🎁  >>>")

 *** Start at: 20:02:30 2025-04-16  ************************************************************
  0:00:00.000  ₀⡄₀₀⡄₀₀.₀₀₀ len prompt 6608
         0:00  ₀⡄₀₀⡄₀₀.₀₀₀ 01. 📄 Файл: lectures_2025/Общественное движение в России 2четверь XIX в_01_large.txt — 5580 символов | Модель: models/mistral-7b-instruct-v0.1.Q8_0.gguf
Prompt 📃📃📃
 [INST] 
            Ты — помощник лектора, готовишь конспект. Преобразуй приложенный текст в структурированный учебный конспект в формате markdown. Новый слайд выделяй символом #
            
            Структура конспекта:
            1.  Заголовок презентации:  из первой строки файла
            
            Разделы: 
            2. План занятия с тайм кодами, 
            3. Основные концепции в формате Название - определение, 
            4. Ключевые персоны c указанием годов жизни в скобках и описанием исторического вклада. 5. Проверяй в своей базе фамилии и имена участников, относящихся к описываемым  
            5. Сравнение концепций c описанием идей

datetime.datetime(2025, 4, 16, 20, 12, 48, 427636)